In [ ]:
%pip -q install requests pandas camelot-py pdfplumber opencv-python-headless

In [ ]:
import re
import io
import requests
import pandas as pd
from pathlib import Path

SOURCE_PDF = (
    "https://fssai.gov.in/upload/advisories/2022/03/"
    "62414c50d2342Direction_RDA_03_02_2022.pdf"
)
OUT_DIR = Path("fever_nutrition_output")
OUT_DIR.mkdir(exist_ok=True)
PDF_PATH = OUT_DIR / "icmr_nin_rda_2020_fssai_annexure.pdf"
CSV_PATH = OUT_DIR / "fever_nutritional_requirements.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36",
    "Accept": "application/pdf,*/*",
}

In [ ]:
response = requests.get(SOURCE_PDF, headers=HEADERS, timeout=90)
response.raise_for_status()

if not response.content.startswith(b"%PDF"):
    raise ValueError("The source URL did not return a PDF file.")

PDF_PATH.write_bytes(response.content)
print(f"Downloaded {len(response.content):,} bytes to {PDF_PATH}")

In [ ]:
import camelot

tables = []
errors = []

for flavor in ("stream", "lattice"):
    try:
        found = camelot.read_pdf(str(PDF_PATH), pages="all", flavor=flavor)
        print(f"{flavor}: detected {found.n} tables")
        if found.n:
            tables = list(found)
            break
    except Exception as exc:
        errors.append((flavor, str(exc)))
        print(f"{flavor} extraction issue: {exc}")

if not tables:
    raise RuntimeError(
        "No PDF tables were detected. See extraction errors above. "
        "If Camelot reports a Ghostscript dependency issue, install Ghostscript "
        "on your system and restart the notebook kernel."
    )

for i, table in enumerate(tables):
    print(f"Table {i}: shape={table.df.shape}")
    display(table.df.head(5))

In [ ]:
def clean_cell(value):
    if pd.isna(value):
        return ""
    return re.sub(r"\\s+", " ", str(value)).strip()

def find_rda_table(camelot_tables):
    for idx, t in enumerate(camelot_tables):
        raw = t.df.copy().applymap(clean_cell)
        flat = " ".join(raw.astype(str).values.flatten()).lower()
        required = ["protein", "calcium", "magnesium", "iron", "zinc", "thiamine", "riboflavin"]
        if all(term in flat for term in required) and ("age group" in flat or "men" in flat):
            return idx, raw
    raise ValueError(
        "Could not identify the ICMR-NIN summary table in extracted tables. "
        "Inspect the printed table previews and adjust the source/table selection."
    )

table_index, raw_table = find_rda_table(tables)
print("Selected table index:", table_index)
display(raw_table.head(12))

In [ ]:
source_columns = [
    "Age Group", "Category of work", "Body Wt (kg)", "Protein (g/d)",
    "Dietary Fiber (g/d)", "Calcium (mg/d)", "Magnesium (mg/d)",
    "Iron (mg/d)", "Zinc (mg/d)", "Iodine (µg/day)", "Thiamine (mg/d)",
    "Riboflavin (mg/d)", "Niacin (mg/d)", "Vit B6 (mg/d)",
    "Folate (µg/d)", "Vit B12 (µg/d)", "Vit C (mg/d)",
    "Vit A (µg/d)", "Vit D (IU/d)"
]

header_row_idx = None
for i in range(min(len(raw_table), 8)):
    row_text = " ".join(raw_table.iloc[i].tolist()).lower()
    if "age group" in row_text and "protein" in row_text:
        header_row_idx = i
        break

if header_row_idx is None:
    print("Header row was not confidently identified. First rows:")
    display(raw_table.head(8))
    raise ValueError("Could not locate the table header row.")

data = raw_table.iloc[header_row_idx + 1:].copy()
data = data.iloc[:, :len(source_columns)]
data.columns = source_columns[:data.shape[1]]

for col in source_columns:
    if col not in data.columns:
        data[col] = ""

data = data[source_columns].copy()

for col in source_columns:
    data[col] = data[col].replace("", pd.NA)
data["Age Group"] = data["Age Group"].ffill()
data["Body Wt (kg)"] = data["Body Wt (kg)"].ffill()
data["Protein (g/d)"] = data["Protein (g/d)"].ffill()
for col in source_columns[5:]:
    data[col] = data[col].ffill()

data["Age Group"] = data["Age Group"].astype(str).str.strip()
data["Category of work"] = data["Category of work"].astype(str).str.strip()
adult = data[
    data["Age Group"].str.fullmatch(r"Men|Women", case=False, na=False)
    & data["Category of work"].str.contains(r"Sedentary|Moderate|Heavy", case=False, na=False)
].copy()

if adult.empty:
    print("Extracted table preview for troubleshooting:")
    display(data.head(20))
    raise ValueError("No adult men/women work-category rows found; inspect the extracted table.")

display(adult[["Age Group", "Category of work", "Protein (g/d)", "Calcium (mg/d)", "Iron (mg/d)"]])

In [ ]:
nutrient_columns = source_columns[3:]

def parse_numeric(cell):
    text = clean_cell(cell).replace(",", "")
    match = re.fullmatch(r"\\s*(\\d+(?:\\.\\d+)?)\\s*", text)
    return float(match.group(1)) if match else None

def split_name_unit(header):
    match = re.match(r"(.+?)\\s*\\((.+?)\\)\\s*$", header)
    if match:
        return match.group(1).strip(), match.group(2).strip()
    return header.strip(), "as stated in source"

records = []
for _, source_row in adult.iterrows():
    sex = "Male" if source_row["Age Group"].lower() == "men" else "Female"
    work = source_row["Category of work"].title()
    for col in nutrient_columns:
        value = parse_numeric(source_row[col])
        if value is None:
            continue
        nutrient, unit = split_name_unit(col)
        records.append({
            "disease": "fever",
            "nutrient": nutrient,
            "min-value": value,
            "max-value": value,
            "unit": unit,
            "age-group": f"Adult ({work})",
            "gender": sex,
        })

result = pd.DataFrame(records, columns=[
    "disease", "nutrient", "min-value", "max-value",
    "unit", "age-group", "gender"
])

if result.empty:
    raise ValueError("No numeric adult nutrient values were extracted.")

result = result.drop_duplicates().reset_index(drop=True)

print("Extracted rows:", len(result))
display(result.head(25))

In [ ]:
result.to_csv(CSV_PATH, index=False, encoding="utf-8")
print("CSV saved to:", CSV_PATH.resolve())
print("Rows:", len(result))
print("Columns:", list(result.columns))

provenance = pd.DataFrame([{
    "source": "FSSAI Annexure IA — Summary of RDA for Indians, ICMR-NIN 2020",
    "source_url": SOURCE_PDF,
    "source_table": "Summary of RDA for Indians",
    "interpretation": (
        "General population RDA values. The source does not specify fever-specific "
        "requirements. 'fever' is only the requested dataset label."
    )
}])
provenance.to_csv(OUT_DIR / "source_provenance.csv", index=False)

result